## Создание презентации с использованием сервиса Presenton

Тестовое обращение через API для создания презентаций с использование сервиса [Presenton](https://presenton.ai/)

In [1]:
import os
from dotenv import load_dotenv
import time
import requests

load_dotenv()

True

In [2]:
slides_num = 2

presentation_context = """
Интеллектуальный агент (ИИ-агент) — это автономная система на базе искусственного интеллекта,
которая самостоятельно получает цель, планирует шаги для её достижения и выполняет действия без постоянного
контроля человека. В отличие от традиционного программного обеспечения или систем роботизации процессов (RPA),
агент гибко адаптируется к ситуации, использует рассуждения, память и доступные инструменты для решения задач.
"""

In [ ]:
def create_presentation(presentation_context, slides_num):
    print('Вспоминаю наш диалог для создания презентации...')
    creating_url = "https://api.presenton.ai/api/v3/presentation/generate/async"
    creating_data = {
        "content": presentation_context,
        "n_slides": slides_num,
        "language": "Russian",
        "standard_template": "modern",
        "export_as": "pptx"
    }
    creating_headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {os.getenv('PRESENTON_API_KEY')}"
    }

    # async with httpx.AsyncClient(timeout=30) as client:
    #     response = await client.post(creating_url, json=creating_data, headers=creating_headers)
    #     result = response.json()
    response = requests.post(creating_url, json=creating_data, headers=creating_headers)
    result = response.json()
    print(result)
    presentation_id = result.get('id', None)

    if presentation_id is None:
        print(f'Ошибка получения id презентации: {result.get('detail', None)}')
        return 1
        
    print('Генерация презентации...')    
    status_url = f"https://api.presenton.ai/api/v3/async-task/status/{presentation_id}"
    status_headers = {"Authorization": f"Bearer {os.getenv('PRESENTON_API_KEY')}"}
    max_retries = 60
    retry_delay = 10

    for _ in range(max_retries):
        # await asyncio.sleep(retry_delay)
        # async with httpx.AsyncClient(timeout=30) as client:
        #     status_resp = await client.get(status_url, headers=status_headers)
        #     status_data = status_resp.json()
        time.sleep(retry_delay)
        status_resp = requests.get(status_url, headers=status_headers)
        status_data = status_resp.json()
        
        cur_status = status_data.get('status', 'failed')
        if cur_status == 'failed':
            print('Ошибка создания презентации')
            return 1 
        if cur_status == 'completed': 
            download_path = status_data.get('data', {}).get("path", None)
            edit_path = status_data.get('data', {}).get("edit_path", None)
            print(f'Презентация готова. Скачать: {download_path}, редактировать: {edit_path}')
            return 0
        

In [4]:
create_presentation(presentation_context, slides_num)

Вспоминаю наш диалог для создания презентации...
{'status': 'pending', 'message': 'Starting standard presentation generation', 'id': 'task-befa11cd864dfd42ff5b58502a767893aa93ecb392159b5e0b534a8065bc63e2', 'data': None, 'error': None, 'created_at': '2026-06-20T07:02:29.083256Z', 'updated_at': '2026-06-20T07:02:29.083259Z'}
Генерация презентации...
Презентация готова. Скачать: https://presenton-public-eu.s3.eu-central-1.amazonaws.com/exports/pptx/----_b8bc7120-a1f9-4c2a-ba2a-4a74abaaabc5.pptx, редактивровать: https://presenton.ai/presentation?id=f97bfd5e-05f4-497e-bcc0-e3238fd634c4&type=standard


0